# Chapter 6 &mdash; Complementation: Swap $F$ and $Q-F$, but Totalize First

**Concept 1 of the Chapter 6 decomposition:** *Complementation of DFA: Swap Final and Non-Final, but Totalize First*

Flipping finality complements the language &mdash; but only on a totalized DFA, or black holes become accepting.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Complementation/Concept-Complementation.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


To complement a DFA, **swap final and non-final states**: $F' = Q - F$.

The catch is **totality**. If $\delta$ is partial, a string that "falls off" the
machine is rejected by *neither* set of final states, so the flip does not complement
anything. Worse, if you totalize *after* flipping, the black hole is created
**non-final** and stays non-final &mdash; and strings that should now be accepted are not.

So the order is fixed: **totalize, then flip.** Jove's `comp_dfa` does both.

## 2. Definitions

even0 = md2mc('''DFA
IF  : 0 -> Od
IF  : 1 -> IF
Od  : 0 -> IF
Od  : 1 -> Od
''')

A two-state machine, already total over $\{0,1\}$.

In [ ]:
even0 = md2mc('''DFA
IF  : 0 -> Od
IF  : 1 -> IF
Od  : 0 -> IF
Od  : 1 -> Od
''')
print("Sigma:", sorted(even0["Sigma"]), " F:", sorted(even0["F"]))

### A deliberately **partial** machine, to see the trap

In [ ]:
partial = md2mc('''DFA
IF : 0 -> Od
Od : 0 -> IF
''')
wide = addtosigma_dfa(partial, {'1'})
print("wide is total?", len(wide["Delta"]) == len(wide["Q"]) * len(wide["Sigma"]))

### Flip by hand, so the two orders can be compared

In [ ]:
def flip(D):
    E = dict(D); E["F"] = D["Q"] - D["F"]; return E

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;14.&nbsp;Running and Testing DFA in Jove with `nthnumeric`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Testing-With-Nthnumeric/Concept-Testing-With-Nthnumeric.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;2.&nbsp;The Product Construction: Union and Intersection of DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Product-Construction/Concept-Product-Construction.ipynb)&nbsp;&rarr;

---

## 3. Tests

On a **total** machine, flipping complements the language exactly.

In [ ]:
comp = comp_dfa(even0)
from itertools import product
strs = [''.join(p) for k in range(9) for p in product('01', repeat=k)]
assert all(accepts_dfa(comp, s) == (not accepts_dfa(even0, s)) for s in strs)
print("complement verified on all %d strings up to length 8" % len(strs))
print("F before:", sorted(even0["F"]), " F after:", sorted(comp["F"]))

On a **partial** machine, flipping first gives the WRONG answer.

In [ ]:
wrong = totalize_dfa(flip(wide))     # flip, THEN totalize  <- bug
right = comp_dfa(wide)               # totalize, THEN flip  <- correct
bad = [s for s in strs if accepts_dfa(wrong, s) != (not accepts_dfa(totalize_dfa(wide), s))]
print("flip-then-totalize disagrees with the true complement on:", bad[:6])
assert bad, "the wrong order really is wrong"
print("\nshortest witness:", repr(bad[0]),
      "-- it falls into the black hole, which flip-first left NON-final")

The correct order agrees with the definition.

In [ ]:
tot = totalize_dfa(wide)
assert all(accepts_dfa(right, s) == (not accepts_dfa(tot, s)) for s in strs)
print("totalize-then-flip: correct on all %d strings" % len(strs))
print("black hole is FINAL in the complement?",
      [q for q in right["F"] if q not in wide["Q"]])

Double complement returns the original.

In [ ]:
print("comp(comp(D)) == D ?", langeq_dfa(comp_dfa(comp_dfa(even0)), even0))
assert langeq_dfa(comp_dfa(comp_dfa(even0)), even0)

## 4. Animation

The complement machine: same shape, opposite double circles.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(comp_dfa(even0), FuseEdges=True)

## 5. Exercises


1. Complement a DFA with **no** final states. What do you get?
2. Why does complementation not work this way for NFA? (Chapter 7.)
3. Write the one-line proof that $\overline{\overline{L}} = L$ for DFA.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Complementation')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')